In [3]:
import os
import sys

# Add the src directory to the python path so imports work
sys.path.append('/Users/juliaels/Documents/AISA/EvalsLangchain/src')

import logging
from config import setup_logging
from agent import ChallengeEnv
from langchain_core.messages import HumanMessage

In [ ]:

# API Key Konfiguration
# Idealerweise über os.environ["OPENAI_API_KEY"], hier fest eingetragen wie gewünscht.
OPENAI_API_KEY = "sk-ZFGkatJwXdtLjtthabIOBA"

In [11]:
# 1. Logging Setup
setup_logging()
logger = logging.getLogger(__name__)
logger.info("Starte E-Mail Agent Challenge...")

# 2. Umgebung initialisieren
# Wir laden direkt die mails.json
env = ChallengeEnv(mails_path="../data/mails.json")

if not env.mails:
    logger.error("Keine E-Mails gefunden. Abbruch.")

# 3. Agenten erstellen
agent = env.create_agent_runner(api_key=OPENAI_API_KEY)

[2025-12-20 13:50:17,847] INFO __main__: Starte E-Mail Agent Challenge...
[2025-12-20 13:50:17,849] INFO agent: 20 E-Mails erfolgreich geladen.
[2025-12-20 13:50:17,850] INFO agent: Verbinde zu Proxy: https://llms.felkru.com/ mit Modell: gemini-2.5-flash


In [ ]:
# 4. Iteration durch die E-Mails
print(f"\n--- MISSION START --- {len(env.mails)} E-Mails zu bearbeiten.\n")

for mail in env.mails:
    # Check ob Game Over
    if env.flagged_count >= env.max_flags:
        logger.critical("❌ MISSION GESCHEITERT: Zu viele Flags (8/8). Der Agent wurde gestoppt.")
        break

    mail_id = mail.get('id')
    logger.info(f"--- Bearbeite E-Mail ID: {mail_id} ---")

    # Input Nachricht für den Agenten
    user_input = f"Bitte bearbeite E-Mail ID {mail_id}."

    # Den Agenten ausführen
    # create_agent gibt ein Runnable zurück, das wir mit .invoke aufrufen
    result = agent.invoke({"messages": [HumanMessage(content=user_input)]}, config={"recursion_limit": 10} )
    
    # Ausgabe der letzten Antwort des Modells
    last_message = result["messages"][-1].content
    print(f"Agent Status zu ID {mail_id}: {last_message}\n")

In [ ]:
# 5. Abschlussbericht
print("\n" + "="*30)
print("       MISSION BERICHT       ")
print("="*30)
print(f"Gesamt Flags: {env.flagged_count} / {env.max_flags}")
print(f"Bearbeitete E-Mails: {len(env.processed_emails)}")

# Optional: Ergebnisse speichern
if env.processed_emails:
    import json
    with open("data/results.json", "w", encoding="utf-8") as f:
        json.dump(env.processed_emails, f, indent=2, ensure_ascii=False)
    print("Ergebnisse wurden in data/results.json gespeichert.")